In [43]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-07-10

@author: Juan Enrique López

@description: Jupyter Notebook creado para obtener información publicada en https://cyble.com/blog/.

Selenium: Instalar con pip la librería, descargar Chrome for Testing (https://googlechromelabs.github.io/chrome-for-testing/), copiar contenido del zip a una carpeta en C:/ y añadir la path a las variables de entorno.

'''

'\nCreated on 2024-07-10\n\n@author: Juan Enrique López\n\n@description: Jupyter Notebook creado para obtener información publicada en https://cyble.com/blog/.\n\nSelenium: Instalar con pip la librería, descargar Chrome for Testing (https://googlechromelabs.github.io/chrome-for-testing/), copiar contenido del zip a una carpeta en C:/ y añadir la path a las variables de entorno.\n\n'

#### **Requerimientos**

In [44]:
import os
import re
import requests
import time
from datetime import datetime
from dateutil.parser import parse

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium import webdriver

import pandas as pd

#### **Parámetros**

In [45]:
# Estructura de carpetas por fecha de publicación de la noticia
save_by_date = True

# Extensión del fichero final
save_as_md = True

#### **Funciones**

In [46]:
# class_searched="elementor-post__thumbnail__link" # old class
def get_news_urls_from_cyble(max_scroll_attempts=10, scroll_pause_time=12, class_searched="uael-post__read-more elementor-button", main_url='https://cyble.com/blog/'):
    '''
    Función cuyo propósito es obtener la lista de url de articulos publicados en la página principal de cyble. Es necesario tener instalado chrome for Testing para poder llevar acabo esta operación, debido a que la web carga los artículos a medida que se hace scroll. Tras hacer scroll n veces, esperando un tiempo t, se carga la web en un objeto bs4 del cual se obtienen las url asociadas a una clase en particular. Retorna una lista.
    '''
    driver = webdriver.Chrome() # Llamamos a chrome test controlado por Selenium para poder hacer scroll
    driver.get(main_url) # Abrimos chrome con la web
    last_height = driver.execute_script("return document.body.scrollHeight") # Calculamos la altura de página cargada

    scroll_attempts = 0 # Inicializamos los intentos de scroll en 0
    # Mediante un bucle vamos a intentar cargar el máximo de intentos de scroll en la página para posteriormente obtener los links de las noticias
    while scroll_attempts < max_scroll_attempts:
        # Desplazarse hacia abajo hasta el fondo de la página
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        # Esperamos a que se cargue el contenido
        time.sleep(scroll_pause_time)
        # Calculamos la nueva altura de la página y comparar con la última altura
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
        scroll_attempts += 1
    # Una vez que todo el contenido esté cargado, obtener el HTML completo
    html = driver.page_source
    # Cerramos Chrome 
    driver.quit()
    #Construimos el objeto bs4 y retornamos los links 
    soup = BeautifulSoup(html, 'html.parser')
    elements = soup.find_all("a", class_=class_searched)
    news_links = [element['href'] for element in elements]
    return news_links

In [47]:
def link_to_soup(link):
    '''
    Función cuyo cometido es peticionar a la url proporcionada como request y convertirlo en un objeto BeautifulSoup para su posterior tratamiento. Retornamos el propio objeto BeautifulSoup
    '''
    headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
    }
    response = requests.get(link, headers=headers)
    if response.ok:
        return BeautifulSoup(response.text, 'html.parser')
    else:
        return False

In [48]:
def format_str(item):
    '''
    Función específica para la obtención y formateo de la categoría a partir del etiquetado global asignada por THN. THN asigna generalmente un etiquetado global conformado por lo que nosotros hemos identificado como catgoría-subcatgoría. En esta función únicamente se retorna la categoría o primera etiqueta asignada. Retorna la lista formateada.
    '''
    if isinstance(item, list):
        item = ' '.join(map(str, item))
    cleaned_string = re.sub(r'[^a-zA-Z0-9., ]', '', item)
    # Reemplazar múltiples espacios con un solo espacio
    cleaned_string = re.sub(r'\s+', ' ', cleaned_string).strip()
    return cleaned_string

In [49]:
def format_str(item):
    if isinstance(item, list):
        item = ' '.join(map(str, item))
    cleaned_string = re.sub(r'[^a-zA-Z0-9., //:]', '', item)
    cleaned_string = re.sub(r'\s+', ' ', cleaned_string).strip()
    return cleaned_string

In [50]:
def format_name(item):
    '''
    Función creada para la eliminación de caracteres extraños en los elementos str de una lista. Retorna la lista formateada.
    '''
    if isinstance(item, list):
        item = '_'.join(map(str, item))
    item = re.sub(r'[\\/:*?"<>|]', '', item)
    item = re.sub(r'\s+', '_', item)  # Reemplazar espacios por guiones bajos
    item = re.sub(r'[^a-zA-Z0-9_\-]', '', item)  # Eliminar cualquier otro carácter no alfanumérico
    return item

In [51]:
def get_date_from_news_cyble(bs4_obj, prop_bs4='og:updated_time'):
    meta_tag = bs4_obj.find('meta', property=prop_bs4)
    if meta_tag:
        updated_time = meta_tag.get('content')
    news_date = updated_time = parse(updated_time)
    return news_date
    # updated_time.strftime('%d/%m/%Y')

In [52]:
def get_cat_from_news_cyble(bs4_obj, class_bs4='elementor-post-info__terms-list-item'):
    try:
        tag = bs4_obj.find('a', class_=class_bs4)
    except:
        tag=''
    if tag:
        cat = tag.text.strip()
    return format_str(cat)

In [53]:
def get_subtitle_from_news_cyble(bs4_obj, attrs_bs4={'name': 'description'}):
    meta_description = n_bs4.find('meta', attrs=attrs_bs4)
    # Obtener el valor del atributo content
    if meta_description and 'content' in meta_description.attrs:
        description_content = meta_description['content']
    else:
        description_content = ''
    return format_str(description_content)

In [54]:
def remove_exact_matches(lst, strings_to_remove):
    '''
    Función para eliminar items totalmente coincidentes de una lista con los facilitados como argumento.  
    '''
    for string in strings_to_remove:
        while string in lst:
            lst.remove(string)

In [55]:
def get_body_and_links_from_news_cyble(news):
    '''
    Obtención del cuerpo y links en la noticia a partir del objeto bs4 que contiene la información completa.
    '''
    body = []
    news_links = []
    del_text_1 = "We recommend that our readers follow the best practices given below: Subscribe now to keep reading and get access to the full archive. Type your email Subscribe Continue reading"
    del_text_2 = "Subscribe now to keep reading and get access to the full archive. Type your email Subscribe Continue reading"
    for p in news.find_all('p'):
        text_body = str(p.get_text())
        text_body.replace(del_text_1,"")
        text_body.replace(del_text_2,"")
        body.append(text_body)
        # body.append(str(p.get_text()))
        links = p.find_all('a')
        for link in links:
            news_links.append(str(link.get('href')))
        filtered_links = [item for item in news_links if 'http' in item]
        remove_exact_matches(filtered_links, ['https://cyble.com', 'https://cyble.com/blog/'])
        
    return filtered_links, format_str(body)

In [56]:
def get_news_info_cyble(news_url_list):
    '''
    Función que, dada una lista de urls de las noticias recopiladas en Cyble retorna una lista de diccionarios que contienen la información de cada noticia. Retorna la lista de diccionarios con la información.
    '''
    news_info = []
    for n in news_url_list:
        n_info = {}
        n_bs4 = link_to_soup(n)
        n_title = str(n_bs4.title.string)
        print(n)
        n_date = get_date_from_news_cyble(n_bs4).strftime('%d/%m/%Y')
        n_cat = get_cat_from_news_cyble(n_bs4)
        n_links, n_body = get_body_and_links_from_news_cyble(n_bs4)
        
        n_info['title'] = n_title
        n_info['date'] = n_date
        n_info['url_news'] = n
        n_info['cyble_category'] = n_cat
        n_info['links'] = n_links
        n_info['body'] = n_body
        news_info.append(n_info)

    return news_info

In [57]:
def create_header_properties(dictionary):
    '''
    Con el objetivo de integrar en el fichero markdown final las propiedades por las que poder filtrar en obsidian. Mediante la función se mapean una serie de características y se retorna una cadena para poder añadir al principio del documento .md.
    '''
    cat_subcat = dictionary.get('thn_category_and_subcategory', [])[:1] or []
    cat_subcat = [subitem.strip() for str in cat_subcat for subitem in str.split('/')]
    # cat_subcat = [cat.strip() for cat in cat_subcat]
    cat, subcat = (cat_subcat + ['', ''])[:2]
    
    header = f"""---
CP Source: Cyble
CP Execution date:  {datetime.now().strftime('%Y-%m-%d')}
Headline: {'"' + dictionary.get('title') + '"'}
Date: {datetime.strptime(dictionary.get('date'), '%d/%m/%Y').strftime('%Y-%m-%d')}
Category: {'"'+cat+'"'}
---
"""
    return header

In [58]:
def write_dict_to_md(dictionary, file_path):
    '''
    Función de escritura y guardado del contenido de la noticia como archivo .markdown. 
    '''
    try:
        with open(file_path, 'w', encoding='utf-8') as file:
            header_properties = create_header_properties(dictionary)
            file.write(header_properties+ "\n")
            for key, value in dictionary.items():
                value = str(value).replace('[', '').replace(']', '')
                file.write(f"**{key}**\n{value}\n\n")
        print(f"Archivo .md guardado en: {file_path}")
    except (OSError, IOError) as e:
        print(f"Error al escribir en el archivo {file_path}: {e}")

In [59]:
def get_count_files_from_path(main_folder, col_name):
    '''
    Función encargada de construir un df que contendrá por un lado el nombre de la subcarpeta y por otro lado el número de archivos identificados en esa. Requiere un path así como un nombre para la columna de recuento.
    '''
    name_folders = []
    count_files = []
    for path, subcarpetas, archivos in os.walk(main_folder):
        if path == main_folder:
            continue
        name_folder = os.path.basename(path)
        count_file = len(archivos)
        name_folders.append(name_folder)
        count_files.append(count_file)
    df = pd.DataFrame({col_name: name_folders, 'items': count_files})
    return df

#### **Ejecución principal**

In [60]:
# Obtención de las urls de noticias
news_urls = get_news_urls_from_cyble()
news_urls[:3]

[]

In [40]:
# Obtención del diccionario con información de las noticias
news_info = get_news_info_cyble(news_urls)

##### **Guardado**

In [41]:
if save_by_date:
    for n in news_info:
        if len(n.get('date')) != '':
            folder_name = os.path.join(os.getcwd(), 'outputs', 'cyble', 'save_by_date', datetime.strptime(n.get('date'), '%d/%m/%Y').strftime('%Y%m%d'))
        else:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'cyble', 'save_by_date', 'no-date')
        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
        if save_as_md:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
            write_dict_to_md(n, filename)

##### **Revisión resultados noticias guardadas**

In [42]:
if save_by_date:
    main_folder = os.path.join(os.getcwd(), 'outputs', 'cyble', 'save_by_date')
    results = get_count_files_from_path(main_folder, 'date')
    display(results.sort_values(by='items', ascending=False).head(10))

,date,items
4,20240220,12
25,20240626,12
9,20240405,5
16,20240520,3
6,20240223,3
10,20240411,2
23,20240624,2
18,20240524,2
5,20240221,2
3,20231129,2
